# Making PyTorch fast: torch.compile, mixed precision, DataLoader, and profiling

_PyTorch (a complete course)_

**Profile first, then speed up: one-line torch.compile, mixed precision, a well-fed GPU, and fewer CPU/GPU syncs.**

Training speed is a pipeline: load a batch on the CPU &rarr; copy it to the GPU &rarr; run forward and
       backward on the GPU. The whole thing runs at the speed of its slowest, un-overlapped stage. Making PyTorch fast
       is mostly about removing stalls so every stage stays busy.

       Two complementary moves: do less work (lower precision via AMP, a fused/compiled graph via
       torch.compile) and waste less time (overlap data loading with compute, avoid host/device
       syncs, use a big enough batch). And before any of it: profile, so you spend effort on the stage that is
       actually slow.

---

This notebook is a practice scaffold for the **Making PyTorch fast: torch.compile, mixed precision, DataLoader, and profiling** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.profiler import profile, record_function, ProfilerActivity

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------------------------------------------
# 0. FREE WIN for FIXED input sizes: let cuDNN pick its fastest
#    convolution algorithm and cache it. (Only helps when the
#    input shape does not change between steps.)
# ------------------------------------------------------------
torch.backends.cudnn.benchmark = True

# A tiny synthetic dataset + model so this runs fast on free Colab.
N, D, C = 4096, 256, 10
X = torch.randn(N, D)
y = torch.randint(0, C, (N,))
model = nn.Sequential(nn.Linear(D, 512), nn.ReLU(), nn.Linear(512, C)).to(device)
loss_fn = nn.CrossEntropyLoss()                 # expects raw logits + class indices
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

# ============================================================
# 1. PROFILE FIRST. Measure before you optimize. Record a few
#    steps and print the top ops by total CUDA (or CPU) time.
# ============================================================
def one_step(xb, yb):
    xb = xb.to(device, non_blocking=True)       # async copy (needs pin_memory)
    yb = yb.to(device, non_blocking=True)
    opt.zero_grad(set_to_none=True)             # set_to_none is cheaper than zeroing
    with record_function("forward"):
        out = model(xb)
        loss = loss_fn(out, yb)
    with record_function("backward"):
        loss.backward()
    opt.step()
    return loss

xb, yb = X[:256], y[:256]
acts = [ProfilerActivity.CPU] + ([ProfilerActivity.CUDA] if device == "cuda" else [])
with profile(activities=acts, record_shapes=True) as prof:
    for _ in range(5):                          # a handful of steps is enough
        one_step(xb, yb)
sort_key = "cuda_time_total" if device == "cuda" else "cpu_time_total"
print(prof.key_averages().table(sort_by=sort_key, row_limit=8))   # <- the bottleneck

# ============================================================
# 2. torch.compile: one line, JIT-compiles + fuses the graph.
#    First call compiles (slow); later calls reuse the graph.
# ============================================================
model = torch.compile(model)                    # PyTorch 2.x
_ = one_step(xb, yb)                            # triggers the one-time compile
print("compiled and warmed up")

# ============================================================
# 3. EFFICIENT DATALOADER: keep the GPU fed.
#    num_workers > 0 prefetches batches in parallel; pin_memory
#    enables fast, overlappable non_blocking transfers.
# ============================================================
loader = DataLoader(
    TensorDataset(X, y),
    batch_size=64,
    shuffle=True,
    num_workers=4,            # subprocesses prepare batches while the GPU computes
    pin_memory=(device == "cuda"),   # page-locked host memory -> fast async copies
    prefetch_factor=2,        # each worker stages 2 batches ahead
    persistent_workers=True,  # don't respawn workers every epoch
    drop_last=True,           # fixed batch shape -> no torch.compile recompiles
)

# ============================================================
# 4. GRADIENT ACCUMULATION: large EFFECTIVE batch on small memory.
#    accum_steps micro-batches of 64 act like one batch of 64*accum.
#    KEY: divide each micro-batch loss by accum_steps (average, not sum).
# ============================================================
accum_steps = 4               # effective batch = 64 * 4 = 256
opt.zero_grad(set_to_none=True)
running = torch.zeros((), device=device)        # accumulate ON-DEVICE: no per-step sync

for i, (xb, yb) in enumerate(loader):
    xb = xb.to(device, non_blocking=True)
    yb = yb.to(device, non_blocking=True)
    loss = loss_fn(model(xb), yb) / accum_steps  # scale so accumulation = average
    loss.backward()                              # grads ACCUMULATE across micro-batches
    running += loss.detach()                     # stays on GPU; do NOT .item() here
    if (i + 1) % accum_steps == 0:               # step once per accum_steps micro-batches
        opt.step()
        opt.zero_grad(set_to_none=True)
    if i >= 4 * accum_steps:                      # short demo run
        break

# Sync to the CPU ONCE, only when we actually need the number to print.
print("avg micro-batch loss:", (running / (i + 1)).item())

## Visualize the data & results

_How much faster does each optimization make training? Training throughput (samples/sec) for baseline vs +AMP vs +torch.compile vs +more DataLoader workers, with each speedup stacking on the previous. ILLUSTRATIVE numbers from a small reproducible model._

In [ ]:
import numpy as np

# Reproducible, ILLUSTRATIVE model of how throughput stacks.
# Each optimization multiplies the previous throughput by a plausible
# per-step speedup factor (not a measured benchmark on your hardware).
baseline_sps = 2500.0          # samples/sec for an un-optimized training loop
speedups = {                   # each factor stacks on the running total
    "+ AMP":            1.80,  # mixed precision: faster per-op (half precision)
    "+ torch.compile":  1.40,  # fused/compiled graph: fewer kernel launches
    "+ more workers":   1.15,  # DataLoader no longer starves the GPU
}

labels = ["baseline"]
values = [baseline_sps]
cur = baseline_sps
for name, factor in speedups.items():
    cur *= factor              # speedups MULTIPLY (different pipeline stages)
    labels.append(name)
    values.append(round(cur))

for lab, v in zip(labels, values):
    print(f"{lab:18s} {v:8.0f} samples/sec")
print("end-to-end speedup:", round(values[-1] / values[0], 2), "x")
# -> baseline 2500, +AMP 4500, +torch.compile 6300, +more workers 7245 (2.9x)

import matplotlib.pyplot as plt
plt.bar(labels, values, color=["#8b949e", "#4ea1ff", "#7ee787", "#f2cc60"])
plt.ylabel("throughput (samples / sec)")
plt.title("Training throughput stacks up with each optimization (illustrative)")
plt.show()

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Your model trains at 30% GPU utilization (per nvidia-smi) and an epoch is slow. Your DataLoader uses num_workers=0 and you copy each batch with plain batch.to("cuda"). What is the bottleneck, and what two changes fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Read the symptom: low GPU utilization with a slow epoch. — _A starved GPU sits idle waiting for the next batch &mdash; the bottleneck is data loading, not compute._
- Note num_workers=0 means the main process loads batches serially with no overlap. — _With zero workers the GPU waits while the CPU prepares each batch; nothing runs in parallel._
- Set num_workers>0 and pin_memory=True, and copy with batch.to("cuda", non_blocking=True). — _Worker subprocesses prefetch the next batches while the GPU computes the current one; pinned memory plus non_blocking lets the copy overlap too._

**Answer:** The GPU is starving for data &mdash; low utilization with a slow epoch is the signature of a data-loading bottleneck, and num_workers=0 confirms it (batches load serially on the main process). Fix it two ways: (1) raise num_workers (e.g. to 4&ndash;8) and set pin_memory=True in the DataLoader so batches are prefetched in parallel into page-locked memory; (2) copy with batch.to("cuda", non_blocking=True) so the transfer overlaps compute. GPU utilization should jump and the epoch should shrink. See pt-data for the full pipeline.

</details>

**Problem 2.** A teammate added model = torch.compile(model) expecting a speedup, but training got slower and the log shows repeated "recompiling" messages. Their batch size varies every step (they drop the last partial batch sometimes, pad differently others). What is happening and how do they fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall torch.compile specializes the compiled graph on the input shapes it sees. — _The fast path is a graph compiled for specific shapes; a new shape is a cache miss._
- Connect the varying batch size to the "recompiling" messages. — _Each new batch shape triggers a fresh, expensive compilation, so they pay the compile cost over and over instead of reusing it._
- Make shapes stable (fixed batch size, drop_last=True, pad to a constant length) or pass dynamic=True. — _Stable shapes let one compiled graph be reused; dynamic=True compiles a shape-flexible graph that tolerates variation._

**Answer:** torch.compile compiles a graph specialized to the input shapes. Because the batch size keeps changing, every new shape is a cache miss that triggers another expensive recompile &mdash; so the model spends its time compiling, not training. Fix: keep shapes constant (fix the batch size, set drop_last=True, pad sequences to a fixed length), or compile with torch.compile(model, dynamic=True) to get one shape-flexible graph. Then the compile cost is paid once and the speedup shows up.

</details>

**Problem 3.** You want an effective batch size of 256 but only 64 fits in GPU memory. Without buying more hardware, how do you train as if the batch were 256, and what is the one detail people get wrong?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize this is what gradient accumulation is for. — _Summing gradients over several small (micro) batches before stepping reproduces the gradient of one large batch._
- Run 4 micro-batches of 64, calling .backward() each (grads accumulate) and optimizer.step() + zero_grad() only after the 4th. — _Because 4 × 64 = 256, four accumulated micro-batch gradients equal one 256-sample gradient._
- Divide each micro-batch loss by 4 (the accumulation count) before .backward(). — _Without the division the accumulated gradient is the sum, not the average, so it is 4× too large &mdash; this is the detail people forget._

**Answer:** Use gradient accumulation: process 4 micro-batches of 64 (since $4\times64=256$), call loss.backward() after each so gradients accumulate, and only call optimizer.step() then optimizer.zero_grad() after the 4th. The detail people get wrong: scale each micro-batch loss by 1/4 before .backward() &mdash; otherwise you accumulate the sum of four gradients instead of the average of a 256-batch, making the effective gradient (and learning rate) 4&times; too large. With the division, it matches a true batch of 256.

</details>